# 11 Model outputs for the dashboard

Writes three files the dashboard reads (`chalseokorom/ai-law-datacenter-siting`, README "Model
outputs the dashboard expects") to `processed/model_outputs/`:

| File | Columns |
|---|---|
| `predictions.csv` | `fips`, `prob_base`, `prob_state`, `prob_aug`, `driver_1`-`driver_3` |
| `clusters.csv` | `fips`, `cluster`, `cluster_name`, `entropy` |
| `metrics.json` | model, one sentence finding, PR-AUC and Brier per feature set (policy row last) |
| `pnnl.csv` | `fips`, `pnnl_modeled`, `n_projected_ref`, `n_scenarios_sited` (refreshes PNNL layer) |

Probabilities are notebook 06's **out of fold, calibrated XGBoost predictions** averaged over three seeds. `prob_state` is + Geography set
(Census division) w/ state grouped validation a held-out state has no state dummy of its own, so the
division is the policy free location control. Drivers are the three largest "out of fold" SHAP values
for + AI policy model.

In [ ]:
import json
import numpy as np
import pandas as pd
from pipeline_config import DIVISIONS, FEATURE_SETS, GEO, OUTPUTS, ROOT, STAGE4, STAGE5, TARGET, load_model_table

d = load_model_table().set_index("county_fips")
oof = pd.read_csv(STAGE4 / "oof_predictions.csv", dtype={"county_fips": str})
scores = pd.read_csv(STAGE4 / "fold_scores.csv")
FULL = FEATURE_SETS["+ AI policy"]

## Predictions & drivers
Builds the per county prediction table the dashboard reads. Each county’s probabilities (infrastructure only, +geography, +AI policy) & a plain language sentence for top SHAP drivers.

In [ ]:
p = oof[oof.model == "XGBoost"].pivot_table(index="county_fips", columns="set", values="prob", aggfunc="mean")
pred = pd.DataFrame({"prob_base": p["Infrastructure"], "prob_state": p["+ Geography"], "prob_aug": p["+ AI policy"]})

# (phrase when county's value high, phrase when low). SHAP sign says whether raised/lowered odds
PHRASE = {
    "land_value_dollar_per_sqft_mean": ("High land value", "Low land value"),
    "distance_to_substation_mean": ("Far from substations on average", "Close to substations on average"),
    "distance_to_substation_min": ("No land right next to a substation", "Some land right next to a substation"),
    "composite_siting_suitability_mean": ("Much of the county suits a data center", "Little land suits a data center"),
    "composite_siting_suitability_max": ("Has land that suits a data center", "No land that suits a data center"),
    "commercial_electricity_rates_dollar_per_kwh_mean": ("High commercial power price", "Low commercial power price"),
    "industrial_electricity_rates_dollar_per_kwh_mean": ("High industrial power price", "Low industrial power price"),
    "real_property_tax_rate_mean": ("High real property tax", "Low real property tax"),
    "personal_property_tax_rate_mean": ("State taxes business equipment (personal property)", "No personal property tax"),
    "sales_tax_rate_mean": ("High sales tax", "Low sales tax"),
    "mechanical_cooling_fraction_mean": ("Hot or dry climate: cooling must be mechanical", "Climate allows water cooling"),
    "pol_bills_log": ("State introduced many AI bills", "State introduced few AI bills"),
    "pol_enacted_log": ("State enacted many AI bills", "State enacted few AI bills"),
    "pol_pass_rate": ("High AI bill pass rate in the state", "Low AI bill pass rate in the state"),
    "pol_first_year": ("State came to AI legislation late", "State legislated on AI early"),
    "pol_share_regulatory": ("State AI bills often regulate companies", "State AI bills rarely regulate companies"),
    "pol_share_public": ("State AI bills often cover government use", "State AI bills rarely cover government use"),
    "pol_share_rights": ("State AI bills often cover rights and safety", "State AI bills rarely cover rights and safety"),
    "pol_infra_bills": ("State has energy, tax or data center AI bills", "No energy, tax or data center AI bills"),
    "pol_infra_enacted": ("State enacted an energy, tax or data center AI bill", "No energy, tax or data center AI bill enacted"),
    **{g: (f"Located in the {name} division", f"Outside the {name} division") for g, name in zip(GEO, DIVISIONS)}}

assert set(FULL) <= set(PHRASE), set(FULL) - set(PHRASE)

shap_oof = pd.read_csv(STAGE4 / "oof_shap.csv", dtype={"county_fips": str}).set_index("county_fips")[FULL]
# Whether a high value of each feature pushes the odds up (+1) or down (-1) across all counties (notebook 07).
# A county's phrase follows SHAP sign, so "high land value" always pairs w/ "raises." Phrase says
# which side of model's turning point county sits on, not how it compares w/ the median.
direction = np.sign(pd.read_csv(STAGE4 / "shap_importance.csv", index_col=0).direction.fillna(1))


def drivers(fips):
    s = shap_oof.loc[fips]
    out = []
    for feat in s.abs().sort_values(ascending=False).index[:3]:
        high = d.at[fips, feat] == 1 if feat in GEO else (s[feat] > 0) == (direction[feat] > 0)
        out.append(f"{PHRASE[feat][0 if high else 1]}: {'raises' if s[feat] > 0 else 'lowers'} the likelihood")
    return out


drv = pd.DataFrame([drivers(f) for f in pred.index], index=pred.index, columns=["driver_1", "driver_2", "driver_3"])
predictions = pred.join(drv).rename_axis("fips").reset_index()
assert len(predictions) == len(d) and predictions[["prob_base", "prob_state", "prob_aug"]].notna().all().all()

predictions.round(4).to_csv(OUTPUTS / "predictions.csv", index=False)
print(predictions.describe().round(3))
predictions.sort_values("prob_aug", ascending=False).head(8)

       prob_base  prob_state  prob_aug
count   3099.000    3099.000  3099.000
mean       0.081       0.082     0.081
std        0.114       0.114     0.115
min        0.000       0.000     0.000
25%        0.026       0.021     0.025
50%        0.042       0.040     0.034
75%        0.073       0.082     0.085
max        0.917       0.969     0.959


,fips,prob_base,prob_state,prob_aug,driver_1,driver_2,driver_3
1333,27123,0.750000,0.547619,0.959097,High land value: raises the likelihood,Close to substations on average: raises the li...,High real property tax: raises the likelihood
2224,42045,0.785998,0.733349,0.925926,High land value: raises the likelihood,Close to substations on average: raises the li...,High real property tax: raises the likelihood
2247,42091,0.799960,0.871136,0.925926,High land value: raises the likelihood,Close to substations on average: raises the li...,High real property tax: raises the likelihood
2191,41051,0.570612,0.574879,0.904762,High land value: raises the likelihood,Close to substations on average: raises the li...,Low commercial power price: lowers the likelihood
2237,42071,0.598962,0.657660,0.833333,High land value: raises the likelihood,Much of the county suits a data center: raises...,Close to substations on average: raises the li...
1743,34023,0.916667,0.968774,0.722222,High land value: raises the likelihood,Close to substations on average: raises the li...,Much of the county suits a data center: raises...
2805,51059,0.496848,0.516676,0.716052,High land value: raises the likelihood,Close to substations on average: raises the li...,Low commercial power price: lowers the likelihood
2431,47093,0.589438,0.537714,0.709804,High land value: raises the likelihood,Much of the county suits a data center: raises...,Close to substations on average: raises the li...


>Predicted probabilities range from near 0 to high 0.9s across all three versions (mean ~8%). Top 8 counties by full (+AI policy) model are shown w/ their three biggest drivers.

## Archetypes

K means labels and names from ntbk 10. `entropy` is K means distance ratio (distance to the nearest
center ÷ distance to the second nearest, 0 = at a center, 1 = halfway btw. two), which is
uncertainty that matches K means labels on map.

In [ ]:
cl = pd.read_csv(STAGE5 / "clusters.csv", dtype={"county_fips": str})
clusters = pd.DataFrame({"fips": cl.county_fips, "cluster": cl.kmeans, "cluster_name": cl.cluster_name,
                         "entropy": cl.kmeans_fit_ratio.round(3),
                         "has_data_center": cl.county_fips.map(d[TARGET])})     # extra column, share w/ data center per archetype

clusters.to_csv(OUTPUTS / "clusters.csv", index=False)
clusters.groupby(["cluster", "cluster_name"]).size()

cluster  cluster_name           
0        Tax-light, pricey land      739
1        Hot or dry, grid-remote     651
2        Grid-close, cheap land     1709
dtype: int64

>The three archetypes split counties into Tax light, pricey land (739), Hot or dry, grid remote (651), and Grid close, cheap land (1,709).

## PNNL comparison fields

From ntbk 08 (exact Census 2024 boundaries), so dashboard's "Versus PNNL" layer uses same 72
reference scenario counties as report.

In [ ]:
pn = pd.read_csv(ROOT / "processed" / "stage6_pnnl" / "pnnl_county.csv", dtype={"county_fips": str})
pn = pn.rename(columns={"county_fips": "fips"})[["fips", "pnnl_modeled", "n_projected_ref", "n_scenarios_sited"]]
pn.assign(pnnl_modeled=pn.pnnl_modeled.astype(int)).to_csv(OUTPUTS / "pnnl.csv", index=False)
print(f"{int(pn.pnnl_modeled.sum()):,} counties in PNNL modeled states; {(pn.n_projected_ref > 0).sum()} chosen in reference scenario")

2,604 counties in PNNL-modeled states; 72 chosen in the reference scenario


>2,604 counties in states PNNL modeled, of those 72 chosen in PNNL’s reference scenario. Same 72 used for comparison in report.

## Scorecard and headline
Final headline numbers for dashboard. **XGBoost**’s scores by feature set, **paired policy** test, and the **permutation** test.

In [ ]:
xgb = scores[scores.model == "XGBoost"].groupby("set")[["pr_auc", "brier"]].mean()
paired = pd.read_csv(STAGE4 / "paired_policy_test.csv", index_col=0).loc["XGBoost"]
perm = pd.read_csv(STAGE4 / "permutation_summary.csv").set_index("family").loc["XGBoost"]
gain, base_rate = paired.policy_gain, d[TARGET].mean()


finding = (f"No. State AI policy moved PR-AUC by only {gain:+.3f} beyond knowing the region, inside fold-to-fold "
           f"noise (±{paired.policy_gain_sd:.3f}) and not reliably better than policy shuffled between states. "
           "Land value and grid access carry the signal.")

metrics = {
    "model": "XGBoost, calibrated",
    "finding": finding,
    "base_rate": round(float(base_rate), 6),
    "permutation_p": round(float(perm.p_value), 3),
    "scores": [{"features": label, "pr_auc": round(float(xgb.loc[s, "pr_auc"]), 6), "brier": round(float(xgb.loc[s, "brier"]), 6)}
               for label, s in [("Infrastructure only", "Infrastructure"), ("Infrastructure + region", "+ Geography"),
                                ("Infrastructure + region + AI policy", "+ AI policy")]]}

(OUTPUTS / "metrics.json").write_text(json.dumps(metrics, indent=2)) # Output
# Check
print(json.dumps(metrics, indent=2))

{
  "model": "XGBoost, calibrated",
  "finding": "No. State AI policy moved PR-AUC by only +0.009 beyond knowing the region, inside fold-to-fold noise (\u00b10.030) and not reliably better than policy shuffled between states. Land value and grid access carry the signal.",
  "base_rate": 0.079058,
  "permutation_p": 0.109,
  "scores": [
    {
      "features": "Infrastructure only",
      "pr_auc": 0.361856,
      "brier": 0.063446
    },
    {
      "features": "Infrastructure + region",
      "pr_auc": 0.372358,
      "brier": 0.063266
    },
    {
      "features": "Infrastructure + region + AI policy",
      "pr_auc": 0.381284,
      "brier": 0.062476
    }
  ]
}


>Trying to have simple sentence explanations